In [1]:
import pathlib as pl
import pickle as pck
import pandas as pd

cfg_nb = pl.Path("../../load-config.ipynb").resolve(strict=True)
%run $cfg_nb

_NB_SESSION = __session__
NB_NAME = pl.Path(_NB_SESSION).name
NB_PATH = pl.Path(_NB_SESSION).parent
NB_REL_PATH = NB_PATH.relative_to(CONFIG["project_repo"]).joinpath(NB_NAME)

sample_sheet_file_assm = CONFIG["project_repo"].joinpath("samples", "verkko_chrY_arhie_freeze.tsv")
sample_sheet_assm = pd.read_csv(sample_sheet_file_assm, sep="\t", header=0, comment="#")

flagger_hifi_path = CONFIG["local_hilbert_prefix"].joinpath(
    CONFIG["flagger_hifi_annotation"]
).resolve(strict=True)

flagger_ont_path = CONFIG["local_hilbert_prefix"].joinpath(
    CONFIG["flagger_ont_annotation"]
).resolve(strict=True)
if "hifi" in str(flagger_ont_path):
    print("REMINDER / WARNING --- flagger ONT still linked to HIFI results for debugging")

nucflag_hifi_path = CONFIG["local_hilbert_prefix"].joinpath(
    CONFIG["nucflag_hifi_annotation"]
).resolve(strict=True)

nucflag_ont_path = CONFIG["local_hilbert_prefix"].joinpath(
    CONFIG["nucflag_ont_annotation"]
).resolve(strict=True)

centromere_path = CONFIG["local_hilbert_prefix"].joinpath(
    CONFIG["centromeres"]
).resolve(strict=True)

kmer_errors_path = CONFIG["local_hilbert_prefix"].joinpath(
    CONFIG["kmer_errors"]
).resolve(strict=True)


def find_all_bed_files(folder):

    bed_files = list(folder.glob("**/*.bed"))
    if not bed_files:
        raise FileNotFoundError(f"No bed files id'd: {folder}")
    print(str(folder)[-50:], " N=", len(bed_files))
    return sorted(bed_files)
        

def extract_sample_id(qc_files, qc_track):

    sample_file = []
    for filepath in qc_files:
        filename = filepath.name
        if qc_track in ["nucflag_hifi", "nucflag_ont"]:
            sample = filename.split(".")[0]
            assert "nucflag" in filename
        elif qc_track in ["flagger_hifi", "flagger_ont", "kmer_errors"]:
            if qc_track == "flagger_hifi":
                assert "flagger/hifi_v2" in str(filepath)
            if qc_track == "flagger_ont":
                assert "flagger/ont_v1" in str(filepath)
            sample = filename.split("_")[0]
        else:
            raise ValueError(filepath)
        sample_file.append((sample, replace_path_prefix(filepath, local_to_remote=True)))
    return sample_file


def extract_centromere_coordinates(centromere_table):

    df = pd.read_csv(centromere_table, sep="\t", header=None, names=["seq", "start", "end"])
    df["sample"] = df["seq"].str.split("_", expand=True)[0]
    assert df["sample"].nunique() == df.shape[0]
    cen_windows = dict()
    for row in df.itertuples():
        window = f"{row.seq}:{row.start}-{row.end}"
        cen_windows[row.sample] = window
    return cen_windows


FORCE_REBUILD_CACHE = False

cache_file_path = prep_cache_file(NB_REL_PATH, "qc_annot_sex_chrom.pkl")
if not cache_file_path.is_file() or FORCE_REBUILD_CACHE:

    annot_files = {
        "nucflag_hifi": find_all_bed_files(nucflag_hifi_path),
        "nucflag_ont": find_all_bed_files(nucflag_ont_path),
        "flagger_hifi": find_all_bed_files(flagger_hifi_path),
        "flagger_ont": find_all_bed_files(flagger_ont_path),
        "kmer_errors": find_all_bed_files(kmer_errors_path),
        "centromere": centromere_path.joinpath(
            "centromeres_live_HOR_array.bed"
        ).resolve(strict=True)
    }

    with open(cache_file_path, "wb") as cache:
        _ = pck.dump(annot_files, cache)
else:
    with open(cache_file_path, "rb") as cache:
        annot_files = pck.load(cache)
    
annot_files = {
    "nucflag_hifi": dict(extract_sample_id(annot_files["nucflag_hifi"], "nucflag_hifi")),
    "nucflag_ont": dict(extract_sample_id(annot_files["nucflag_ont"], "nucflag_ont")),
    "flagger_hifi": dict(extract_sample_id(annot_files["flagger_hifi"], "flagger_hifi")),
    "flagger_ont": dict(extract_sample_id(annot_files["flagger_ont"], "flagger_ont")),
    "kmer_errors": dict(extract_sample_id(annot_files["kmer_errors"], "kmer_errors")),
    "centromere": extract_centromere_coordinates(annot_files["centromere"])
}

sample_sheet_assm["nucflag_hifi"] = sample_sheet_assm["sample"].replace(annot_files["nucflag_hifi"])
sample_sheet_assm["nucflag_ont"] = sample_sheet_assm["sample"].replace(annot_files["nucflag_ont"])
sample_sheet_assm["flagger_hifi"] = sample_sheet_assm["sample"].replace(annot_files["flagger_hifi"])
sample_sheet_assm["flagger_ont"] = sample_sheet_assm["sample"].replace(annot_files["flagger_ont"])
sample_sheet_assm["kmer_errors"] = sample_sheet_assm["sample"].replace(annot_files["kmer_errors"])
sample_sheet_assm["centromere"] = sample_sheet_assm["sample"].replace(annot_files["centromere"])

# debug while waiting for remaining flagger output
keep_samples = (
    sample_sheet_assm["nucflag_hifi"].str.endswith(".bed")
    &
    sample_sheet_assm["nucflag_ont"].str.endswith(".bed")
    &
    sample_sheet_assm["flagger_hifi"].str.endswith(".bed")
    &
    sample_sheet_assm["flagger_ont"].str.endswith(".bed")
    &
    sample_sheet_assm["kmer_errors"].str.endswith(".bed")
)
sample_sheet_assm = sample_sheet_assm.loc[keep_samples, :].copy()
sample_sheet_assm.set_index("sample", inplace=True)

assert not sample_sheet_assm.empty

try:
    # decision from 2025-11-18
    # drop sample HG03456 because of XYY karyotype
    sample_sheet_assm.drop("HG03456", axis=0, inplace=True)
except KeyError:
    # fine as well, implicityl dropped
    # because not annotated
    pass

sample_sheet_file = CONFIG["project_repo"].joinpath("samples", "verkko_chrY_arhie_freeze.assm-qc.tsv")
with open(sample_sheet_file, "w") as table:
    _ = table.write(f"# {TIMESTAMP}\n")
    _ = table.write(f"# N={sample_sheet_assm.shape[0]}\n")
    for group, count in sample_sheet_assm["sample_group"].value_counts().items():
        _ = table.write(f"# group {group}: N={count}\n")
    sample_sheet_assm.to_csv(table, sep="\t", header=True, index=True)


-incoming/upenn/keith-oshima/20250922_nucflag_v034  N= 143
-incoming/upenn/keith-oshima/20251203_nucflag_v100  N= 143
ng/ucsc/prajna-hebbar/flagger/hifi_v2/whole_genome  N= 143
ing/ucsc/prajna-hebbar/flagger/ont_v1/whole_genome  N= 143
ncoming/nhgri/arang-rhie/merqury_qv/error_kmer_bed  N= 143
